# Análise de Cotas de EEB, LR e Escoamento das Bacias

Este script processa arquivos geográficos para compilar dados das bacias hidrográficas e alternativas de linhas de recalque.

Entrada esperada:
- bacias
- coluna_bacias: nome da coluna do shape de bacias onde tem o nome das bacias
- eeb
- MDE.tif
- caminho_lr: linha de recalque

Saída (definido pelo usuário):
- Para cada bacia: cota da EEB, maior cota ao longo de cada linha de recalque, extensão da linha, e bacia de destino da linha.

O script associa automaticamente os dados às bacias e usa o MDE (com fallback online) para calcular elevações.

In [1]:
import numpy as np
import geopandas as gpd
import rasterio
import requests
from shapely.geometry import Point, LineString
from shapely.geometry.multilinestring import MultiLineString
from geopandas.tools import sjoin
import os
from pyproj import Transformer

## Funções Auxiliares

In [8]:
def extrair_cota_com_fallback(ponto, src_principal, src_fallback=None, vetor_crs="EPSG:31982"):
    if ponto is None or ponto.is_empty:
        return np.nan

    # 1. Preparar coordenadas (extrai do Point ou MultiPoint)
    if ponto.geom_type == 'MultiPoint':
        pt_x, pt_y = ponto.geoms[0].x, ponto.geoms[0].y
    else:
        pt_x, pt_y = ponto.x, ponto.y

    def buscar_no_raster(src):
        try:
            # Reprojetar para o CRS do raster atual
            transformer = Transformer.from_crs(vetor_crs, src.crs, always_xy=True)
            x_mde, y_mde = transformer.transform(pt_x, pt_y)
            
            # src.sample recebe uma lista de tuplas [(x, y)]
            for val in src.sample([(x_mde, y_mde)]):
                v = val[0]
                # Verifica se é nodata, NaN ou valor absurdo (comum em MDTs)
                if v == src.nodata or np.isnan(v) or v <= -999:
                    return None
                return float(v)
        except:
            return None
        return None

    # Tenta no Principal
    cota = buscar_no_raster(src_principal)

    # Se falhou, tenta no Fallback
    if cota is None and src_fallback is not None:
        cota = buscar_no_raster(src_fallback)

    return cota if cota is not None else np.nan


def extrair_cota_dem_fallback(ponto, src_fallback, vetor_crs="EPSG:31982"):
    """
    Extrai a cota (elevação) de um ponto a partir do MDE de fallback (baixa resolução).

    Argumentos:
        ponto (shapely.geometry.Point ou MultiPoint): ponto a ser avaliado (no CRS dos vetores).
        src_fallback (rasterio.DatasetReader): MDE de fallback aberto.
        vetor_crs (str ou CRS): CRS dos vetores.

    Retorna:
        float: cota extraída ou np.nan em caso de falha.
    """
    if ponto is None or ponto.is_empty:
        return np.nan

    try:
        # Coordenadas no CRS dos vetores
        if ponto.geom_type == 'MultiPoint':
            x, y = ponto.geoms[0].x, ponto.geoms[0].y
        elif ponto.geom_type == 'Point':
            x, y = ponto.x, ponto.y
        else:
            return np.nan  # Tipo não suportado

        # Reprojetar para o CRS do raster de fallback
        transformer = Transformer.from_crs(vetor_crs, src_fallback.crs, always_xy=True)
        x_dem, y_dem = transformer.transform(x, y)

        row, col = src_fallback.index(x_dem, y_dem)
        valor = src_fallback.read(1, window=((row, row+1), (col, col+1)))[0, 0]

        if (hasattr(src_fallback, "nodata") and valor == src_fallback.nodata) or np.isnan(valor):
            return np.nan

        return float(valor)

    except Exception:
        return np.nan

def interpolar_linha_em_pontos_v0(geom, n_amostras=1000):
    """
    Interpola pontos ao longo de uma geometria do tipo LineString ou MultiLineString.
    Converte MultiLineString em LineString contínua e gera uma amostragem uniforme.

    Argumentos:
        geom (shapely.geometry.LineString ou MultiLineString): geometria da linha.
        n_amostras (int): número mínimo de amostras ao longo da linha.

    Retorna:
        list[Point]: lista de pontos interpolados ao longo da linha.
    """
    if geom is None or geom.is_empty:
        return []

    if geom.geom_type == 'MultiLineString':
        coords = []
        for line in geom.geoms:  # ✅ corrigido
            coords.extend(line.coords)
        geom = LineString(coords)

    if geom.geom_type != 'LineString':
        return []

    n = max(n_amostras, int(geom.length))
    return [geom.interpolate(i / n, normalized=True) for i in range(n + 1)]

def interpolar_linha_em_pontos(geom, intervalo_m=10.0):
    if geom is None or geom.is_empty:
        return []
    if geom.geom_type == "MultiLineString":
        coords = []
        for line in geom.geoms:
            coords.extend(line.coords)
        geom = LineString(coords)
    if geom.geom_type != "LineString":
        return []
    L = geom.length
    if L == 0:
        return [Point(geom.coords[0])]
    dists = np.arange(0.0, L, float(intervalo_m))
    pontos = [geom.interpolate(float(d), normalized=False) for d in dists]
    if pontos and pontos[-1].distance(Point(geom.coords[-1])) > 1e-9:
        pontos.append(Point(geom.coords[-1]))
    return pontos
    
def extrair_atributos_bacia(geometria, bacias, nome):
    """
    Identifica a bacia que contém a geometria fornecida e retorna o valor da coluna `nome`.
    """
    if geometria is None or geometria.is_empty:
        return None

    if geometria.geom_type in ['LineString', 'MultiLineString']:
        if geometria.geom_type == 'LineString':
            ponto = Point(geometria.coords[0])
        else:
            ponto = Point(geometria.geoms[0].coords[0])
    else:
        ponto = geometria

    for _, bacia in bacias.iterrows():
        if ponto.within(bacia.geometry):
            return bacia[nome]

    return None


def carregar_shapefile_com_bacia(caminho, bacias,nome):
    """
    Lê um shapefile de pontos ou linhas e associa cada geometria à bacia correspondente.

    Adiciona duas colunas ao GeoDataFrame:
    - 'nome_bacia': nome da bacia onde a geometria está inserida.
    - 'etapa': etapa correspondente à bacia.

    Argumentos:
        caminho (str): caminho do shapefile a ser carregado.
        bacias (GeoDataFrame): bacias poligonais de referência.

    Retorna:
        GeoDataFrame: shapefile com colunas adicionais de bacia.
    """
    gdf = gpd.read_file(caminho)
    gdf['nome_bacia'] = gdf['geometry'].apply(lambda geom: extrair_atributos_bacia(geom, bacias, nome))
    return gdf

from tqdm.auto import tqdm  # funciona bem no Jupyter, VS Code e terminal

def maior_cota_lr(lrs, mde_path, mde_reserva_path, vetor_crs="EPSG:31982",
                  show_progress=True, show_point_progress=False):
    """
    Calcula a maior cota ao longo de cada linha de recalque (LR) com base no MDE,
    usando fallback para um segundo MDE, e mostrando barras de progresso com tqdm.

    Args:
        lrs (GeoDataFrame): linhas de recalque com coluna 'nome_bacia' e geometria.
        mde_path (str): caminho para o arquivo raster (MDE principal).
        mde_reserva_path (str): caminho para o MDE de fallback (baixa resolução).
        vetor_crs (str): CRS dos vetores (ex.: 'EPSG:31982').
        show_progress (bool): exibe barra para as LRs.
        show_point_progress (bool): exibe barra aninhada para os pontos de cada LR.

    Retorna:
        GeoDataFrame
    """
    with rasterio.open(mde_path) as src_principal, rasterio.open(mde_reserva_path) as src_fallback:
        resultados = []

        it_lrs = lrs.iterrows()
        if show_progress:
            it_lrs = tqdm(it_lrs, total=len(lrs), desc="Linhas de recalque", unit="LR")

        for _, lr in it_lrs:
            geom = lr.geometry
            pontos = interpolar_linha_em_pontos(geom)

            it_pts = pontos
            if show_point_progress:
                it_pts = tqdm(pontos, desc="Pontos da LR", unit="pt", leave=False)

            cotas = [
                extrair_cota_com_fallback(
                    p,
                    src_principal,
                    src_fallback=src_fallback,
                    vetor_crs=vetor_crs
                )
                for p in it_pts
            ]

            if all(np.isnan(cotas)):
                max_cota = np.nan
                max_ponto = None
            else:
                max_idx = int(np.nanargmax(cotas))
                max_cota = float(cotas[max_idx])
                max_ponto = pontos[max_idx]

            resultados.append({
                "bacia_orig": lr.get("nome_bacia", None),
                "cota": max_cota,
                "geometry": max_ponto
            })

                # Se não tiver nenhuma LR (lista vazia), devolve um GDF vazio mas válido
        if not resultados:
            return gpd.GeoDataFrame(
                {"bacia_orig": [], "cota": [], "geometry": gpd.GeoSeries([], dtype="geometry")},
                geometry="geometry",
                crs=lrs.crs,
            )

        return gpd.GeoDataFrame(resultados, geometry="geometry", crs=lrs.crs)


def adicionar_cota(gdf, mde_path, mde_reserva_path, vetor_crs="EPSG:31982"):
    """
    Adiciona uma coluna 'cota' a um GeoDataFrame de pontos, com base em um MDE.

    Se a cota não estiver disponível no MDE principal, tenta o MDE de fallback.

    Argumentos:
        gdf (GeoDataFrame): pontos (como EEBs) para os quais será atribuída a elevação.
        mde_path (str): caminho do MDE raster principal.
        mde_reserva_path (str): caminho do MDE raster de fallback.
        vetor_crs (str): CRS dos vetores.

    Retorna:
        GeoDataFrame: com coluna 'cota' preenchida.
    """
    with rasterio.open(mde_path) as src_principal, rasterio.open(mde_reserva_path) as src_fallback:
        gdf['cota'] = gdf['geometry'].apply(
            lambda geom: extrair_cota_com_fallback(
                geom,
                src_principal,
                src_fallback=src_fallback,
                vetor_crs=vetor_crs
            )
        )
    return gdf


def processar_alternativa(df, bacias, lr, pl, sufixo,nome):
    """
    Processa uma alternativa de linha de recalque e integra os dados ao DataFrame principal.

    Para cada bacia de origem:
    - Associa extensão da linha.
    - Cota máxima no trajeto da linha.
    - Bacia de destino (com base no ponto final da linha).

    Argumentos:
        df (DataFrame): tabela principal com as bacias (coluna 'nome').
        bacias (GeoDataFrame): polígonos de bacia.
        lr (GeoDataFrame): linhas de recalque da alternativa.
        pl (GeoDataFrame): pontos de maior cota por LR.
        sufixo (str): identificador da alternativa (ex: 'A1').

    Retorna:
        DataFrame: df atualizado com colunas da alternativa adicionadas.
    """
    def obter_ultimo_ponto(geom):
        if geom is None or geom.is_empty:
            return None
        if geom.geom_type == 'LineString':
            return Point(geom.coords[-1])
        elif geom.geom_type == 'MultiLineString':
            ultima_linha = geom.geoms[-1]
            return Point(ultima_linha.coords[-1])
        else:
            return None

    ultimos_pontos = [obter_ultimo_ponto(geom) for geom in lr.geometry]
    ultimos_gdf = gpd.GeoDataFrame({'bacia_orig': lr['nome_bacia']}, geometry=ultimos_pontos, crs=lr.crs)

    ultimos_com_bacia = sjoin(ultimos_gdf, bacias, how='left', predicate='within')
    ultimos_com_bacia.rename(columns={nome: f'bacia_destino_{sufixo}'}, inplace=True)

    lr['extensao_m'] = lr.geometry.length

    temp_df = pl[['bacia_orig', 'cota']].copy()
    temp_df = temp_df.merge(
        lr[['nome_bacia', 'extensao_m']],
        left_on='bacia_orig', right_on='nome_bacia', how='left'
    ).drop(columns='nome_bacia')

    temp_df = temp_df.merge(
        ultimos_com_bacia[['bacia_orig', f'bacia_destino_{sufixo}']],
        on='bacia_orig', how='left'
    )

    temp_df.columns = ['bacia_orig', f'cota_pl_{sufixo}', f'extensao_lr_{sufixo}', f'bacia_destino_{sufixo}']
    df = df.merge(temp_df, left_on=nome, right_on='bacia_orig', how='left')
    return df.drop(columns=['bacia_orig'])


## Código Principal

In [11]:
# Carrega dados principais
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Ijuí\SHP criado\BACIAS_COM_SB18_SEM_SB02_SB17C.gpkg')
coluna_bacias = 'Nome'
eeb = carregar_shapefile_com_bacia(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Ijuí\SHP criado\EEB_COM_SB18_V3.gpkg',bacias,coluna_bacias)
mde_path = os.path.join(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Ijuí\Material-Jamas\mdt_mosaico.tif')
mde_reserva = os.path.join(r'C:/Users/gabriel.coimbra/Desktop/CORSAN/Ijuí/mdt_reserva_ijui.tif')
caminho_lr = os.path.join(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Ijuí\SHP criado\LR_COM_SB18.gpkg')
saida = r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Ijuí\SaidaCodigo'
crs = "EPSG:31981"

# Define o nome da alternativa e monta o caminho da linha de recalque
alt_nome = 'AlternativaUnicaIjui_23.02.26'

In [12]:
#coloca tudo no mesmo crs
lr = lr.to_crs(bacias.crs)
eeb = eeb.to_crs(bacias.crs)

if not os.path.exists(caminho_lr):
    print(f"Arquivo de linha de recalque não encontrado: {caminho_lr}")

# DataFrame base com bacias
df = bacias[[coluna_bacias]].copy()

# Cotas das EEBs
eeb = adicionar_cota(eeb, mde_path, mde_reserva, vetor_crs=eeb.crs)
eeb_por_bacia = eeb[['nome_bacia', 'cota']].drop_duplicates().rename(columns={'cota': 'cota_eeb'})
df = df.merge(eeb_por_bacia, left_on=coluna_bacias, right_on='nome_bacia', how='left')

# Processa linha de recalque
lr = carregar_shapefile_com_bacia(caminho_lr, bacias,coluna_bacias)
pl = maior_cota_lr(lr, mde_path, mde_reserva, vetor_crs=crs)
df = processar_alternativa(df, bacias, lr, pl, alt_nome,coluna_bacias)

# Salva CSV
df.to_excel(
    os.path.join(saida,f'bacia_destino_{alt_nome}.xlsx'),
    index=False,
)

print("Planilha gerada com sucesso!")

Linhas de recalque:   0%|          | 0/41 [00:00<?, ?LR/s]

Planilha gerada com sucesso!


In [7]:
with rasterio.open(mde_path) as src:
    # Pega a primeira EEB
    ponto_teste = eeb.geometry.iloc[0]
    
    # Reprojeção manual para teste
    transformer = Transformer.from_crs(eeb.crs, src.crs, always_xy=True)
    x_dem, y_dem = transformer.transform(ponto_teste.x, ponto_teste.y)
    
    print(f"CRS EEB: {eeb.crs}")
    print(f"CRS Raster: {src.crs}")
    print(f"Extensão Raster (Bounds): {src.bounds}")
    print(f"Ponto Transformado: X={x_dem}, Y={y_dem}")
    
    # Verifica se o ponto está dentro dos limites do raster
    is_inside = (src.bounds.left <= x_dem <= src.bounds.right) and \
                (src.bounds.bottom <= y_dem <= src.bounds.top)
    print(f"Ponto está dentro da área do MDT? {is_inside}")

CRS EEB: EPSG:31981
CRS Raster: EPSG:31982
Extensão Raster (Bounds): BoundingBox(left=210640.1478, bottom=6852011.0416, right=218640.1478, top=6859011.0416)
Ponto Transformado: X=212003.63648771262, Y=6854770.491220915
Ponto está dentro da área do MDT? True


In [5]:
df = df.drop_duplicates()

df.to_csv(
    os.path.join(saida,f'bacia_destino_dropduplicates.csv'),
    index=False,
)